# **Bike Sharing Systems - Data Wrangling**

This notebook cleans and prepares all datasets collected in the previous step for analysis and modeling.

1. Column name standardization across all datasets using regular expressions
2. Regex cleaning of the scraped bike-sharing systems data, removing reference links from city, system, and operator fields
3. Missing value handling for the Seoul bike demand dataset
4. Categorical variable encoding via indicator (dummy) variables
5. Min-max normalization of numeric columns

Output files feed directly into the EDA and modeling steps of the pipeline.

## **Setup**

In [18]:
library(tidyverse)

## **Standardize Column Names**

Convert all column names to uppercase with underscore separators across all
collected datasets, ensuring consistency for downstream processing.

In [19]:
dataset_list <- c(
  "raw_bike_sharing_systems.csv",
  "raw_seoul_bike_sharing.csv",
  "raw_cities_weather_forecast.csv",
  "raw_worldcities.csv"
)

for (dataset_name in dataset_list) {
  dataset <- read_csv(dataset_name, show_col_types = FALSE)
  names(dataset) <- toupper(names(dataset))
  names(dataset) <- str_replace_all(names(dataset), " ", "_")
  write.csv(dataset, dataset_name, row.names = FALSE)
}

# Verify column names
for (dataset_name in dataset_list) {
  cat("\n", dataset_name, "\n")
  print(names(read_csv(dataset_name, show_col_types = FALSE)))
}


 raw_bike_sharing_systems.csv 
[1] "COUNTRY...1"   "COUNTRY...2"   "CITY_/_REGION" "NAME"         
[5] "SYSTEM"        "OPERATOR"      "LAUNCHED"      "DISCONTINUED" 

 raw_seoul_bike_sharing.csv 
 [1] "DATE"                  "RENTED_BIKE_COUNT"     "HOUR"                 
 [4] "TEMPERATURE"           "HUMIDITY"              "WIND_SPEED"           
 [7] "VISIBILITY"            "DEW_POINT_TEMPERATURE" "SOLAR_RADIATION"      
[10] "RAINFALL"              "SNOWFALL"              "SEASONS"              
[13] "HOLIDAY"               "FUNCTIONING_DAY"      

 raw_cities_weather_forecast.csv 
 [1] "CITY"              "WEATHER"           "VISIBILITY"       
 [4] "TEMP"              "TEMP_MIN"          "TEMP_MAX"         
 [7] "PRESSURE"          "HUMIDITY"          "WIND_SPEED"       
[10] "WIND_DEG"          "FORECAST_DATETIME" "SEASON"           

 raw_worldcities.csv 
 [1] "CITY"       "CITY_ASCII" "LAT"        "LNG"        "COUNTRY"   
 [6] "ISO2"       "ISO3"       "ADMIN_NAME" "CAPITAL"

## **Regex Cleaning - Bike Sharing Systems**

The scraped Wikipedia data contains reference links and textual
annotations in numeric fields.
These are cleaned using regular expressions via `stringr`.

In [20]:
bike_sharing_df <- read_csv("raw_bike_sharing_systems.csv", show_col_types = FALSE)

# Check column names
names(bike_sharing_df)
head(bike_sharing_df)

[1] "COUNTRY...1"   "COUNTRY...2"   "CITY_/_REGION" "NAME"         
[5] "SYSTEM"        "OPERATOR"      "LAUNCHED"      "DISCONTINUED"

COUNTRY...1,COUNTRY...2,CITY_/_REGION,NAME,SYSTEM,OPERATOR,LAUNCHED,DISCONTINUED
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Albania,Albania,Tirana[5],Ecovolis,NA,NA,March 2011,Discontinued
Argentina,Argentina,Buenos Aires[6][7],Ecobici,Serttel Brasil[8],Bike In Baires Consortium[9],2010,NA
Argentina,Argentina,Mendoza[10],Metrobici,NA,NA,2014,NA
Argentina,Argentina,Rosario,Mi Bici Tu Bici[11],NA,NA,2 December 2015,NA
Argentina,Argentina,"San Lorenzo, Santa Fe",Biciudad,Biciudad,NA,27 November 2016,NA
Australia,Australia,Melbourne[12],Melbourne Bike Share,PBSC & 8D,Motivate,June 2010,30 November 2019[13]


In [21]:
# Rename and select relevant columns
result <- bike_sharing_df %>%
  rename(
    COUNTRY = `COUNTRY...1`,
    CITY    = `CITY_/_REGION`
  ) %>%
  select(COUNTRY, CITY, NAME, SYSTEM, OPERATOR, LAUNCHED, DISCONTINUED)

# Detect reference link pattern e.g. [1], [23]
find_reference_pattern <- function(strings) {
  ref_pattern <- "\\[[0-9]+\\]"
  str_detect(strings, ref_pattern)
}

# Remove reference links and trim whitespace
remove_ref <- function(strings) {
  ref_pattern <- "\\[[0-9]+\\]"
  str_trim(str_replace_all(strings, ref_pattern, ""))
}

# Clean reference links from text columns
result <- result %>%
  mutate(
    CITY     = remove_ref(CITY),
    SYSTEM   = remove_ref(SYSTEM),
    OPERATOR = remove_ref(OPERATOR)
  )

# Verify no reference links remain
result %>%
  filter(
    find_reference_pattern(CITY) |
    find_reference_pattern(SYSTEM) |
    find_reference_pattern(OPERATOR)
  )

summary(result)

write.csv(result, "bike_sharing_systems.csv", row.names = FALSE)


COUNTRY,CITY,NAME,SYSTEM,OPERATOR,LAUNCHED,DISCONTINUED
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


   COUNTRY              CITY               NAME              SYSTEM         
 Length:897         Length:897         Length:897         Length:897        
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
   OPERATOR           LAUNCHED         DISCONTINUED      
 Length:897         Length:897         Length:897        
 Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character  

## **Seoul Bike Demand - Missing Values**

The Seoul bike demand dataset requires handling of missing values before
modeling. `RENTED_BIKE_COUNT` is the response variable and cannot have
missing values. `TEMPERATURE` is an important predictor whose missing
values are imputed using the seasonal mean.

In [22]:
bike_sharing_df <- read_csv("raw_seoul_bike_sharing.csv", show_col_types = FALSE)
summary(bike_sharing_df)
dim(bike_sharing_df)

# Drop rows where the response variable is missing
bike_sharing_df <- bike_sharing_df %>%
  filter(!is.na(RENTED_BIKE_COUNT))

dim(bike_sharing_df)

# Inspect missing temperature rows
bike_sharing_df %>%
  filter(is.na(TEMPERATURE))

# All missing temperature values fall in Summer, impute with Summer mean
summer_mean_temp <- bike_sharing_df %>%
  filter(SEASONS == "Summer", !is.na(TEMPERATURE)) %>%
  summarise(mean_temp = mean(TEMPERATURE)) %>%
  pull(mean_temp)

bike_sharing_df <- bike_sharing_df %>%
  mutate(TEMPERATURE = ifelse(
    is.na(TEMPERATURE) & SEASONS == "Summer",
    summer_mean_temp,
    TEMPERATURE
  ))

# Confirm no missing values remain
sum(is.na(bike_sharing_df$TEMPERATURE))


     DATE           RENTED_BIKE_COUNT      HOUR        TEMPERATURE    
 Length:8760        Min.   :   2.0    Min.   : 0.00   Min.   :-17.80  
 Class :character   1st Qu.: 214.0    1st Qu.: 5.75   1st Qu.:  3.40  
 Mode  :character   Median : 542.0    Median :11.50   Median : 13.70  
                    Mean   : 729.2    Mean   :11.50   Mean   : 12.87  
                    3rd Qu.:1084.0    3rd Qu.:17.25   3rd Qu.: 22.50  
                    Max.   :3556.0    Max.   :23.00   Max.   : 39.40  
                    NA's   :295                       NA's   :11      
    HUMIDITY       WIND_SPEED      VISIBILITY   DEW_POINT_TEMPERATURE
 Min.   : 0.00   Min.   :0.000   Min.   :  27   Min.   :-30.600      
 1st Qu.:42.00   1st Qu.:0.900   1st Qu.: 940   1st Qu.: -4.700      
 Median :57.00   Median :1.500   Median :1698   Median :  5.100      
 Mean   :58.23   Mean   :1.725   Mean   :1437   Mean   :  4.074      
 3rd Qu.:74.00   3rd Qu.:2.300   3rd Qu.:2000   3rd Qu.: 14.800      
 Max.   :98.

[1] 8760   14

[1] 8465   14

DATE,RENTED_BIKE_COUNT,HOUR,TEMPERATURE,HUMIDITY,WIND_SPEED,VISIBILITY,DEW_POINT_TEMPERATURE,SOLAR_RADIATION,RAINFALL,SNOWFALL,SEASONS,HOLIDAY,FUNCTIONING_DAY
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
07/06/2018,3221,18,NA,57,2.7,1217,16.4,0.96,0.0,0,Summer,No Holiday,Yes
12/06/2018,1246,14,NA,45,2.2,1961,12.7,1.39,0.0,0,Summer,No Holiday,Yes
13/06/2018,2664,17,NA,57,3.3,919,16.4,0.87,0.0,0,Summer,No Holiday,Yes
17/06/2018,2330,17,NA,58,3.3,865,16.7,0.66,0.0,0,Summer,No Holiday,Yes
20/06/2018,2741,19,NA,61,2.7,1236,17.5,0.60,0.0,0,Summer,No Holiday,Yes
30/06/2018,1144,13,NA,87,1.7,390,23.2,0.71,3.5,0,Summer,No Holiday,Yes
05/07/2018,827,10,NA,75,1.1,1028,20.8,1.22,0.0,0,Summer,No Holiday,Yes
11/07/2018,634,9,NA,96,0.6,450,24.9,0.41,0.0,0,Summer,No Holiday,Yes
12/07/2018,593,6,NA,93,1.1,852,24.3,0.01,0.0,0,Summer,No Holiday,Yes


[1] 0

## **Seoul Bike Demand - Indicator Variables**

Regression models require numeric inputs. Categorical columns `SEASONS`,
`HOLIDAY`, and `FUNCTIONING_DAY` are converted to indicator (dummy) variables.
`HOUR` is treated as categorical despite being numeric, as bike demand
patterns vary significantly by hour of the day.

In [23]:
# Check unique values of FUNCTIONING_DAY after missing value removal
unique(bike_sharing_df$FUNCTIONING_DAY)

# Convert HOUR to character so it is treated as categorical
bike_sharing_df <- bike_sharing_df %>%
  mutate(HOUR = as.character(HOUR))

# Create indicator variables for all categorical columns
bike_sharing_df <- bike_sharing_df %>%
  mutate(dummy = 1) %>%
  pivot_wider(
    names_from  = SEASONS,
    values_from = dummy,
    values_fill = 0,
    names_prefix = ""
  ) %>%
  mutate(dummy = 1) %>%
  pivot_wider(
    names_from  = HOLIDAY,
    values_from = dummy,
    values_fill = 0
  ) %>%
  mutate(dummy = 1) %>%
  pivot_wider(
    names_from  = FUNCTIONING_DAY,
    values_from = dummy,
    values_fill = 0
  ) %>%
  mutate(dummy = 1) %>%
  pivot_wider(
    names_from  = HOUR,
    values_from = dummy,
    values_fill = 0,
    names_prefix = "HOUR_"
  )

summary(bike_sharing_df)

write.csv(bike_sharing_df, "seoul_bike_sharing_converted.csv", row.names = FALSE)


[1] "Yes"

     DATE           RENTED_BIKE_COUNT  TEMPERATURE        HUMIDITY    
 Length:8465        Min.   :   2.0    Min.   :-17.80   Min.   : 0.00  
 Class :character   1st Qu.: 214.0    1st Qu.:  3.00   1st Qu.:42.00  
 Mode  :character   Median : 542.0    Median : 13.50   Median :57.00  
                    Mean   : 729.2    Mean   : 12.77   Mean   :58.15  
                    3rd Qu.:1084.0    3rd Qu.: 22.70   3rd Qu.:74.00  
                    Max.   :3556.0    Max.   : 39.40   Max.   :98.00  
   WIND_SPEED      VISIBILITY   DEW_POINT_TEMPERATURE SOLAR_RADIATION 
 Min.   :0.000   Min.   :  27   Min.   :-30.600       Min.   :0.0000  
 1st Qu.:0.900   1st Qu.: 935   1st Qu.: -5.100       1st Qu.:0.0000  
 Median :1.500   Median :1690   Median :  4.700       Median :0.0100  
 Mean   :1.726   Mean   :1434   Mean   :  3.945       Mean   :0.5679  
 3rd Qu.:2.300   3rd Qu.:2000   3rd Qu.: 15.200       3rd Qu.:0.9300  
 Max.   :7.400   Max.   :2000   Max.   : 27.200       Max.   :3.5200  
    RA

## **Seoul Bike Demand - Normalization**

Numeric columns are on different scales. Min-max normalization rescales
each to the range [0, 1] to prevent large-valued features from dominating
the regression model.

$$x_{new} = \frac{x_{old} - x_{min}}{x_{max} - x_{min}}$$

In [24]:
normalize <- function(x) {
  (x - min(x, na.rm = TRUE)) / (max(x, na.rm = TRUE) - min(x, na.rm = TRUE))
}

numeric_cols <- c(
  "RENTED_BIKE_COUNT", "TEMPERATURE", "HUMIDITY", "WIND_SPEED",
  "VISIBILITY", "DEW_POINT_TEMPERATURE", "SOLAR_RADIATION",
  "RAINFALL", "SNOWFALL"
)

bike_sharing_normalized_df <- bike_sharing_df %>%
  mutate(across(all_of(numeric_cols), normalize))

summary(bike_sharing_normalized_df %>% select(all_of(numeric_cols)))

write.csv(bike_sharing_normalized_df, "seoul_bike_sharing_converted_normalized.csv", row.names = FALSE)


 RENTED_BIKE_COUNT  TEMPERATURE        HUMIDITY        WIND_SPEED    
 Min.   :0.00000   Min.   :0.0000   Min.   :0.0000   Min.   :0.0000  
 1st Qu.:0.05965   1st Qu.:0.3636   1st Qu.:0.4286   1st Qu.:0.1216  
 Median :0.15194   Median :0.5472   Median :0.5816   Median :0.2027  
 Mean   :0.20460   Mean   :0.5345   Mean   :0.5933   Mean   :0.2332  
 3rd Qu.:0.30445   3rd Qu.:0.7080   3rd Qu.:0.7551   3rd Qu.:0.3108  
 Max.   :1.00000   Max.   :1.0000   Max.   :1.0000   Max.   :1.0000  
   VISIBILITY     DEW_POINT_TEMPERATURE SOLAR_RADIATION       RAINFALL       
 Min.   :0.0000   Min.   :0.0000        Min.   :0.000000   Min.   :0.000000  
 1st Qu.:0.4602   1st Qu.:0.4412        1st Qu.:0.000000   1st Qu.:0.000000  
 Median :0.8429   Median :0.6107        Median :0.002841   Median :0.000000  
 Mean   :0.7131   Mean   :0.5977        Mean   :0.161326   Mean   :0.004261  
 3rd Qu.:1.0000   3rd Qu.:0.7924        3rd Qu.:0.264205   3rd Qu.:0.000000  
 Max.   :1.0000   Max.   :1.0000        Ma

## **Standardize Column Names - Processed Datasets**

Apply the same naming convention to the newly created processed datasets.

In [25]:
processed_list <- c(
  "seoul_bike_sharing_converted.csv",
  "seoul_bike_sharing_converted_normalized.csv"
)

for (dataset_name in processed_list) {
  dataset <- read_csv(dataset_name, show_col_types = FALSE)
  names(dataset) <- toupper(names(dataset))
  names(dataset) <- str_replace_all(names(dataset), " ", "_")
  write.csv(dataset, dataset_name, row.names = FALSE)
}
